# Modelling

## Load Packages

In [ ]:
from catboost import CatBoostClassifier
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torchmetrics.classification import BinaryCalibrationError
from xgboost import XGBClassifier
import joblib
import numpy as np
import os
import pandas as pd
import torch

## Set-up

In [ ]:
seed = 123
os.makedirs("../models", exist_ok=True)

## Data Wrangling

In [ ]:
features_dfs = []

for features_file in os.listdir("../data/features"):
    features_dfs.append(pd.read_parquet(f"../data/features/{features_file}"))

labels_dfs = []

for labels_file in os.listdir("../data/labels"):
    labels_dfs.append(pd.read_parquet(f"../data/labels/{labels_file}"))

X = pd.concat(features_dfs)
y = pd.concat(labels_dfs)

key = ["game_id", "action_id"]
X = X.sort_values(key).reset_index(drop=True)
y = y.sort_values(key).reset_index(drop=True)

In [ ]:
for col in X.select_dtypes(include="bool").columns:
    X[col] = X[col].astype("int8")

period_cols = [c for c in X.columns if "period" in c]
X[period_cols] = X[period_cols].astype("int8")

max_pitch_distance = np.sqrt(105**2 + 68**2)

distance_cols = [
    "actor_minimum_distance_to_opponent",
    "actor_minimum_distance_to_teammate",
    "actor_mean_distance_to_opponent",
    "actor_mean_distance_to_teammate",
]

for base_col in distance_cols:
    cols = [c for c in X.columns if c.startswith(base_col)]
    X[cols] = X[cols].fillna(max_pitch_distance)

zero_fill_cols = [
    "actor_stdev_distance_to_opponent",
    "actor_stdev_distance_to_teammate",
    "number_of_opponents_within_radius_of_actor",
    "number_of_teammates_within_radius_of_actor",
    "opponent_convex_hull_area",
    "teammate_convex_hull_area",
]

for base_col in zero_fill_cols:
    cols = [c for c in X.columns if c.startswith(base_col)]
    X[cols] = X[cols].fillna(0)

embedding_cols = [
    c for c in X.columns
    if c.startswith("spatial_embedding_")
]

X[embedding_cols] = X[embedding_cols].fillna(0)

float64_cols = X.select_dtypes(include="float64").columns
X[float64_cols] = X[float64_cols].astype("float32")

In [ ]:
id_cols = ["game_id", "action_id"]

manual_spatial_feature_prefixes = [
    "actor_minimum_distance_to_opponent",
    "actor_minimum_distance_to_teammate",
    "actor_mean_distance_to_opponent",
    "actor_mean_distance_to_teammate",
    "actor_stdev_distance_to_opponent",
    "actor_stdev_distance_to_teammate",
    "number_of_opponents_within_radius_of_actor",
    "number_of_teammates_within_radius_of_actor",
    "opponent_convex_hull_area",
    "teammate_convex_hull_area",
]

manual_spatial_feature_cols = [
    c for c in X.columns
    if any(c.startswith(prefix) for prefix in manual_spatial_feature_prefixes)
]

spatial_embedding_feature_cols = [
    c for c in X.columns
    if c.startswith("spatial_embedding_")
]

baseline_feature_cols = [
    c for c in X.columns
    if (
        c not in manual_spatial_feature_cols
        and c not in spatial_embedding_feature_cols
    )
]

feature_sets = {
    "id_cols": id_cols,
    "baseline": baseline_feature_cols,
    "manual": baseline_feature_cols + manual_spatial_feature_cols,
    "embeddings": baseline_feature_cols + spatial_embedding_feature_cols,
    "all": baseline_feature_cols + manual_spatial_feature_cols + spatial_embedding_feature_cols,
}

y_scores = y[["game_id", "action_id", "scores"]].astype(int)
y_concedes = y[["game_id", "action_id", "concedes"]].astype(int)

In [ ]:
game_ids = X["game_id"].unique()

train_games, test_games = train_test_split(
    game_ids,
    test_size=0.2,
    random_state=seed,
)

train_mask = X["game_id"].isin(train_games)
test_mask = X["game_id"].isin(test_games)

X_train = {
    name: X.loc[train_mask, cols]
    for name, cols in feature_sets.items()
}

X_test = {
    name: X.loc[test_mask, cols]
    for name, cols in feature_sets.items()
}

targets_train = {
    "scores": y_scores.loc[train_mask],
    "concedes": y_concedes.loc[train_mask],
}

targets_test = {
    "scores": y_scores.loc[test_mask],
    "concedes": y_concedes.loc[test_mask],
}

In [ ]:
joblib.dump(X_train, "../data/modelling/train/X_train_dict.pkl")
joblib.dump(X_test, "../data/modelling/test/X_test_dict.pkl")
joblib.dump(targets_train, "../data/modelling/train/targets_train_dict.pkl")
joblib.dump(targets_test, "../data/modelling/test/targets_test_dict.pkl")

del X_train, X_test

In [ ]:
X.sort_values(["game_id", "action_id"], inplace=True)
y.sort_values(["game_id", "action_id"], inplace=True)

## Modelling

In [ ]:
model_defs = {
    "logistic_regression": LogisticRegression(
        solver="lbfgs",
        random_state=seed
    ),

    "random_forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=3,
        random_state=seed
    ),

    "xgboost": XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=seed
    ),

    "catboost": CatBoostClassifier(
        random_seed=seed,
        verbose=False
    ),
}

In [11]:
models = {}

model_dir = "../models/classifiers"
os.makedirs(model_dir, exist_ok=True)

for model_name, model_template in model_defs.items():

    for feature_name, cols in feature_sets.items():

        if feature_name == "id_cols":
            continue

        X_train_set = (
            X.loc[train_mask, cols]
            .drop(columns=id_cols)
        )

        for target_name, y_train_set in targets_train.items():

            print(f"Training {model_name} | {feature_name} | {target_name}")

            model = clone(model_template)
            model.fit(X_train_set, y_train_set.drop(["game_id", "action_id"], axis=1))

            models[(model_name, feature_name, target_name)] = model

            joblib.dump(
                model,
                f"{model_dir}/{model_name}_{feature_name}_{target_name}.pkl",
            )

Training logistic_regression | baseline | scores


c:\Users\Charlie\Documents\coding\python\projects\dissertation\.venv\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\Charlie\Documents\coding\python\projects\dissertation\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Training logistic_regression | baseline | concedes


c:\Users\Charlie\Documents\coding\python\projects\dissertation\.venv\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\Charlie\Documents\coding\python\projects\dissertation\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Training logistic_regression | manual | scores


c:\Users\Charlie\Documents\coding\python\projects\dissertation\.venv\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\Charlie\Documents\coding\python\projects\dissertation\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


... (23 outputs left out)

c:\Users\Charlie\Documents\coding\python\projects\dissertation\.venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Training random_forest | all | concedes


c:\Users\Charlie\Documents\coding\python\projects\dissertation\.venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Training xgboost | baseline | scores
Training xgboost | baseline | concedes
Training xgboost | manual | scores
Training xgboost | manual | concedes
Training xgboost | embeddings | scores
Training xgboost | embeddings | concedes
Training xgboost | all | scores
Training xgboost | all | concedes
Training catboost | baseline | scores
Training catboost | baseline | concedes
Training catboost | manual | scores
Training catboost | manual | concedes
Training catboost | embeddings | scores
Training catboost | embeddings | concedes
Training catboost | all | scores
Training catboost | all | concedes


In [12]:
results = []
bce = BinaryCalibrationError()

for model_name, model_template in model_defs.items():

    for feature_name, cols in feature_sets.items():

        if feature_name == "id_cols": 
            continue

        X_test_set = (
            X.loc[test_mask, cols]
            .drop(columns=id_cols)
        )

        for target_name, y_test_set in targets_test.items():

            model = models[(model_name, feature_name, target_name)]

            probs = model.predict_proba(X_test_set)[:, 1]

            results.append({
                "model": model_name,
                "feature_set": feature_name,
                "target": target_name,
                "brier_score": brier_score_loss(y_test_set.drop(["game_id", "action_id"], axis=1), probs),
                "log_loss": log_loss(y_test_set.drop(["game_id", "action_id"], axis=1), probs),
                "ece": bce(
                    torch.tensor(probs, dtype=torch.float32),
                    torch.tensor(y_test_set.drop(["game_id", "action_id"], axis=1).values.squeeze(), dtype=torch.int64),
                ).item(),
                "auc": roc_auc_score(y_test_set.drop(["game_id", "action_id"], axis=1), probs),
            })

In [13]:
results_df = (
    pd.DataFrame(results)
    .sort_values(
        ["model", "target", "feature_set"]
    )
)

results_df.to_parquet("../results/model_training_results.parquet", compression="zstd", index=False)